# LoTTE IR Evaluation Notebook

تقييم نظام الاسترجاع على **كل qrels queries** من LoTTE.

**قبل التشغيل:**
1. preprocessing + retrieval + query_refinement services شغّالين
2. الفهرس مبني (`data/indexes/lotte_lifestyle_dev_forum/`)

**الخلايا:**
- Cell 2: إحصائيات Dataset
- Cell 3: تشغيل evaluation (أو استخدم report.json الموجود)
- Cell 4–5: Charts MAP / nDCG
- Cell 6–7: مقارنة قبل/بعد Query Refinement

`QUERY_LIMIT = None` للتسليم النهائي. استخدم `100` للاختبار السريع.

In [ ]:
import ir_datasets
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.run_evaluation import run_evaluation
from shared.config import DEFAULT_IR_DATASET
from shared.schemas import RetrievalModel

DATASET = DEFAULT_IR_DATASET
QUERY_LIMIT = None  # None = all qrels queries (2076). Use 100 for quick testing.
TOP_K = 10

In [ ]:
dataset = ir_datasets.load(DATASET)
qrels_count = dataset.qrels_count()
queries_count = dataset.queries_count()
docs_count = dataset.docs_count()

print('=' * 70)
print(f'Dataset: {DATASET}')
print(f'Documents: {docs_count:,}')
print(f'Queries: {queries_count:,}')
print(f'Total queries in qrels file: {qrels_count:,}')
print(f'Queries used in this notebook run: {QUERY_LIMIT or qrels_count:,}')
print('=' * 70)

In [ ]:
MODELS = [
    RetrievalModel.TF_IDF,
    RetrievalModel.BM25,
    RetrievalModel.EMBEDDING,
    RetrievalModel.HYBRID_SERIAL,
    RetrievalModel.HYBRID_PARALLEL,
    RetrievalModel.HYBRID_BRANCHING,
]

report = run_evaluation(
    dataset_name=DATASET,
    models=MODELS,
    top_k=TOP_K,
    query_limit=QUERY_LIMIT,
    output_path=Path('data/evaluation/notebook_report.json'),
)
report

In [ ]:
import pandas as pd

df = pd.DataFrame(report['metrics']).T
df.index.name = 'model'
print(f"Evaluated queries: {report['evaluated_queries']} / {report['total_qrels_queries']}")
df

In [ ]:
from scripts.plot_evaluation_charts import load_report, plot_model_comparison

REPORT_PATH = Path('data/evaluation/report.json')
if not REPORT_PATH.exists():
    REPORT_PATH = Path('data/evaluation/notebook_report.json')
CHARTS_DIR = Path('data/evaluation/charts')

report_for_charts = load_report(REPORT_PATH) if REPORT_PATH.exists() else report
df_metrics = plot_model_comparison(report_for_charts, CHARTS_DIR)
print(f'Charts saved to {CHARTS_DIR}/')
df_metrics

In [ ]:
from IPython.display import Image, display

for name in ['map_comparison.png', 'ndcg_comparison.png', 'combined_metrics.png']:
    path = CHARTS_DIR / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

In [ ]:
from scripts.run_evaluation import run_refinement_comparison

REFINEMENT_MODELS = [RetrievalModel.BM25, RetrievalModel.HYBRID_PARALLEL]
COMPARISON_PATH = Path('data/evaluation/refinement_comparison.json')

comparison = run_refinement_comparison(
    dataset_name=DATASET,
    models=REFINEMENT_MODELS,
    top_k=TOP_K,
    query_limit=QUERY_LIMIT,
    output_path=COMPARISON_PATH,
    use_api=False,
)
comparison

In [ ]:
from scripts.plot_evaluation_charts import plot_refinement_comparison

if COMPARISON_PATH.exists():
    comp = load_report(COMPARISON_PATH)
    plot_refinement_comparison(comp, CHARTS_DIR)
    for name in ['refinement_map.png', 'refinement_ndcg.png']:
        path = CHARTS_DIR / name
        if path.exists():
            print(name)
            display(Image(filename=str(path)))